In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

import utils
from const import *

In [ ]:
half_hourly_usage_df = utils.get_half_hourly_usage_data()
monthly_usage_df = utils.get_monthly_usage_data()

display(half_hourly_usage_df.head(), f"{len(half_hourly_usage_df)} rows")
display(monthly_usage_df.head(), f"{len(monthly_usage_df)} rows")

In [ ]:
duplicated_idx = half_hourly_usage_df[START_DT_COL].duplicated(keep='first')
adjustment_df = half_hourly_usage_df.loc[duplicated_idx].groupby(START_DT_COL)[UNITS_COL].sum()
half_hourly_usage_df = half_hourly_usage_df.loc[~duplicated_idx]
update_idx = half_hourly_usage_df[START_DT_COL].isin(adjustment_df.index)
half_hourly_usage_df.loc[update_idx, UNITS_COL] = half_hourly_usage_df.loc[update_idx, UNITS_COL] + adjustment_df

In [ ]:
half_hourly_usage_df['time'] = half_hourly_usage_df[START_DT_COL].dt.time
half_hourly_usage_df['date'] = half_hourly_usage_df[START_DT_COL].dt.date

In [ ]:
june_2024_units_df = half_hourly_usage_df.loc[
    (half_hourly_usage_df[START_DT_COL].dt.month.isin([6]))
    & (half_hourly_usage_df[START_DT_COL].dt.year == 2024)
    
].pivot(index='time', columns='date', values=UNITS_COL)

display(june_2024_units_df.head())

mu = np.mean(june_2024_units_df.to_numpy(), axis=1)
X_cov = np.cov(june_2024_units_df.to_numpy())

var = np.sqrt(np.diag(X_cov))
X_corr = X_cov/np.outer(var, var)

print(np.linalg.matrix_rank(X_cov))

In [ ]:
fig = px.imshow(
    X_corr,
    x=june_2024_units_df.index,
    y=june_2024_units_df.index,
)
fig.update_layout(
    height=600,
    width=600,
)
fig.show()

fig = px.imshow(
    X_cov,
    x=june_2024_units_df.index,
    y=june_2024_units_df.index,
)
fig.update_layout(
    height=600,
    width=600,
)
fig.show()